In [1]:
# W12D4: End-to-End MLOps Pipeline Review

%pip install -q fastapi uvicorn scikit-learn pytest httpx ruff

print("W12D4 environment setup complete!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 32.8 MB/s eta 0:00:00
W12D4 environment setup complete!


In [2]:
# W12D4: Create project folder and ML API

from pathlib import Path

project_dir = Path("/content/W12D4_End_to_End_MLOps")
project_dir.mkdir(parents=True, exist_ok=True)

app_code = '''
from fastapi import FastAPI, Request
from pydantic import BaseModel
from sklearn.linear_model import LogisticRegression
import time

app = FastAPI(title="W12D4 End-to-End MLOps API")

X = [[0], [1], [2], [3], [4], [5]]
y = [0, 0, 0, 1, 1, 1]

model = LogisticRegression()
model.fit(X, y)

metrics = {
    "total_requests": 0,
    "failed_requests": 0,
    "total_latency_ms": 0.0
}


class PredictionInput(BaseModel):
    value: float


@app.middleware("http")
async def monitor_requests(request: Request, call_next):
    start = time.perf_counter()
    metrics["total_requests"] += 1

    try:
        response = await call_next(request)
        if response.status_code >= 400:
            metrics["failed_requests"] += 1
        return response
    except Exception:
        metrics["failed_requests"] += 1
        raise
    finally:
        metrics["total_latency_ms"] += (
            time.perf_counter() - start
        ) * 1000


@app.get("/health")
def health():
    return {"status": "healthy"}


@app.post("/predict")
def predict(data: PredictionInput):
    prediction = int(model.predict([[data.value]])[0])
    probability = float(model.predict_proba([[data.value]])[0][1])

    return {
        "input": data.value,
        "prediction": prediction,
        "positive_probability": round(probability, 4)
    }


@app.get("/monitoring")
def monitoring():
    total = metrics["total_requests"]

    return {
        **metrics,
        "average_latency_ms": round(
            metrics["total_latency_ms"] / total, 2
        ) if total else 0.0,
        "failure_rate_percent": round(
            metrics["failed_requests"] / total * 100, 2
        ) if total else 0.0
    }
'''

(project_dir / "app.py").write_text(app_code.strip() + "\n")

print("Project folder:", project_dir)
print("Created app.py:", (project_dir / "app.py").exists())

Project folder: /content/W12D4_End_to_End_MLOps
Created app.py: True


In [3]:
# W12D4: Create automated API tests

tests_dir = project_dir / "tests"
tests_dir.mkdir(parents=True, exist_ok=True)

test_code = '''
from fastapi.testclient import TestClient
from app import app

client = TestClient(app)


def test_health():
    response = client.get("/health")
    assert response.status_code == 200
    assert response.json()["status"] == "healthy"


def test_prediction():
    response = client.post("/predict", json={"value": 4})
    assert response.status_code == 200
    assert response.json()["prediction"] == 1
    assert "positive_probability" in response.json()


def test_monitoring():
    response = client.get("/monitoring")
    assert response.status_code == 200
    data = response.json()
    assert "total_requests" in data
    assert "average_latency_ms" in data
    assert "failure_rate_percent" in data
'''

(tests_dir / "test_app.py").write_text(test_code.strip() + "\n")

print("Created tests/test_app.py")
print("Three API tests are ready!")

Created tests/test_app.py
Three API tests are ready!


In [4]:
# W12D4: Run automated tests

import subprocess

result = subprocess.run(
    ["python", "-m", "pytest", "-v"],
    cwd=str(project_dir),
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Tests failed. Share the error so we can fix it.")

print("SUCCESS: All three tests passed!")

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/W12D4_End_to_End_MLOps
plugins: anyio-4.15.1, platformdirs-4.12.2, langsmith-0.14.2, typeguard-4.6.0
collecting ... collected 3 items

tests/test_app.py::test_health PASSED                                    [ 33%]
tests/test_app.py::test_prediction PASSED                                [ 66%]
tests/test_app.py::test_monitoring PASSED                                [100%]

============================== 3 passed in 2.59s ===============================

SUCCESS: All three tests passed!


In [5]:
# W12D4: Create Docker and dependency files

(project_dir / "requirements.txt").write_text(
    "fastapi\n"
    "uvicorn\n"
    "scikit-learn\n"
    "pydantic\n"
)

(project_dir / "requirements-dev.txt").write_text(
    "pytest\n"
    "httpx\n"
    "ruff\n"
)

dockerfile = """
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .

EXPOSE 8000

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
"""

(project_dir / "Dockerfile").write_text(dockerfile.strip() + "\n")

(project_dir / ".gitignore").write_text("""
__pycache__/
*.py[cod]
.pytest_cache/
.ruff_cache/
.venv/
.env
.ipynb_checkpoints/
""".strip() + "\n")

print("Created requirements.txt")
print("Created requirements-dev.txt")
print("Created Dockerfile")
print("Created .gitignore")

Created requirements.txt
Created requirements-dev.txt
Created Dockerfile
Created .gitignore


In [7]:
# W12D4: Document production monitoring and retraining

monitoring_text = """
# W12D4: End-to-End MLOps Monitoring Strategy

## 1. Metrics to Monitor
- API request volume and error rate.
- Average and P95 response latency.
- CPU, memory, and container health.
- Missing, invalid, or out-of-range input features.
- Prediction distribution and model confidence.
- Model quality when ground-truth labels become available.

## 2. Alerts
- Repeated API health-check failures.
- HTTP 5xx error rate above 5% over a sustained window.
- P95 latency exceeding the agreed service target.
- Unexpected input values or significant feature drift.
- Model performance dropping below its approved threshold.

Thresholds should be calibrated to actual service requirements.

## 3. Retraining Triggers
- Confirmed degradation in model performance.
- Persistent data drift supported by investigation.
- Concept drift detected using labelled outcomes.
- Sufficient new, representative training data becoming available.

Data drift alone should trigger investigation, not automatic deployment.

## 4. End-to-End Pipeline
1. Store application code and dependencies in version control.
2. Run linting and automated tests in GitHub Actions.
3. Build a versioned Docker image.
4. Push the image to a registry after credentials are configured.
5. Deploy and monitor the service.
6. Evaluate retrained models before releasing them.

## 5. Safe Model Updates
- Validate new training data.
- Compare candidate model performance with the current model.
- Run automated tests and egression checks.
- Deploy gradually and keep a rollback option.

## 6. Current Project Limitations
Metrics are stored in memory and reset when the process restarts.
Production deployments should export metrics to persistent monitoring
infrastructure and use statistical drift checks appropriate to the data.
"""

(project_dir / "monitoring.md").write_text(
    monitoring_text.strip() + "\n"
)

print("Created monitoring.md")
print("Monitoring, alerting, retraining, and deployment strategy documented!")

Created monitoring.md
Monitoring, alerting, retraining, and deployment strategy documented!


In [8]:
# W12D4: Create GitHub Actions workflow

workflow_dir = project_dir / ".github" / "workflows"
workflow_dir.mkdir(parents=True, exist_ok=True)

workflow = """
name: W12D4 End-to-End MLOps Pipeline

on:
  push:
    branches: ["**"]
  pull_request:
    branches: ["**"]

jobs:
  lint-test-build:
    runs-on: ubuntu-latest

    steps:
      - name: Checkout repository
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt -r requirements-dev.txt

      - name: Lint code
        run: ruff check app.py tests

      - name: Run tests
        run: pytest -v

      - name: Build Docker image
        run: docker build -t w12d4-mlops:${{ github.sha }} .

      - name: Log in to Docker Hub
        if: github.event_name == 'push' && github.ref == 'refs/heads/main'
        uses: docker/login-action@v3
        with:
          username: ${{ secrets.DOCKERHUB_USERNAME }}
          password: ${{ secrets.DOCKERHUB_TOKEN }}

      - name: Push Docker image
        if: github.event_name == 'push' && github.ref == 'refs/heads/main'
        run: |
          docker tag w12d4-mlops:${{ github.sha }} ${{ secrets.DOCKERHUB_USERNAME }}/w12d4-mlops:${{ github.sha }}
          docker push ${{ secrets.DOCKERHUB_USERNAME }}/w12d4-mlops:${{ github.sha }}
"""

workflow_path = workflow_dir / "ci.yml"
workflow_path.write_text(workflow.strip() + "\n")

print("Created:", workflow_path)
print("Workflow exists:", workflow_path.exists())
print("Pipeline stages: lint -> test -> Docker build -> optional registry push")

Created: /content/W12D4_End_to_End_MLOps/.github/workflows/ci.yml
Workflow exists: True
Pipeline stages: lint -> test -> Docker build -> optional registry push


In [9]:
# W12D4: Run tests and save the real output

import subprocess

result = subprocess.run(
    ["python", "-m", "pytest", "-v"],
    cwd=str(project_dir),
    capture_output=True,
    text=True
)

test_output = result.stdout + "\n" + result.stderr
print(test_output)

outputs_dir = project_dir / "outputs"
outputs_dir.mkdir(exist_ok=True)
(outputs_dir / "test_results.txt").write_text(test_output)

if result.returncode != 0:
    raise RuntimeError("Tests failed. Share the error before continuing.")

print("Tests passed and saved to outputs/test_results.txt")

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/W12D4_End_to_End_MLOps
plugins: anyio-4.15.1, platformdirs-4.12.2, langsmith-0.14.2, typeguard-4.6.0
collecting ... collected 3 items

tests/test_app.py::test_health PASSED                                    [ 33%]
tests/test_app.py::test_prediction PASSED                                [ 66%]
tests/test_app.py::test_monitoring PASSED                                [100%]

============================== 3 passed in 1.61s ===============================


Tests passed and saved to outputs/test_results.txt


In [12]:
# W12D4: Load the FastAPI app and save API outputs

import sys
import json
from fastapi.testclient import TestClient

# Make the project folder importable
sys.path.insert(0, str(project_dir))

# Import the app from app.py
from app import app

client = TestClient(app)

# Health check
health_output = client.get("/health").json()

# Prediction
prediction_output = client.post(
    "/predict", json={"value": 4}
).json()

# Monitoring
monitoring_output = client.get("/monitoring").json()

# Save outputs
outputs_dir = project_dir / "outputs"
outputs_dir.mkdir(exist_ok=True)

(outputs_dir / "health_output.json").write_text(
    json.dumps(health_output, indent=2)
)
(outputs_dir / "prediction_output.json").write_text(
    json.dumps(prediction_output, indent=2)
)
(outputs_dir / "monitoring_metrics.json").write_text(
    json.dumps(monitoring_output, indent=2)
)

print("HEALTH:", json.dumps(health_output, indent=2))
print("PREDICTION:", json.dumps(prediction_output, indent=2))
print("MONITORING:", json.dumps(monitoring_output, indent=2))
print("\nAll three outputs saved successfully.")

HEALTH: {
  "status": "healthy"
}
PREDICTION: {
  "input": 4.0,
  "prediction": 1,
  "positive_probability": 0.843
}
MONITORING: {
  "total_requests": 3,
  "failed_requests": 0,
  "total_latency_ms": 4.56489800012605,
  "average_latency_ms": 1.52,
  "failure_rate_percent": 0.0
}

All three outputs saved successfully.


In [13]:
# W12D4: Final code quality and test verification

import subprocess

# Run Ruff lint checks
lint_result = subprocess.run(
    ["python", "-m", "ruff", "check", "app.py", "tests"],
    cwd=str(project_dir),
    capture_output=True,
    text=True
)

lint_output = lint_result.stdout + lint_result.stderr
print("=== LINT RESULTS ===")
print(lint_output or "No lint issues found.")

(outputs_dir / "lint_results.txt").write_text(
    lint_output or "All lint checks passed.\n"
)

if lint_result.returncode != 0:
    raise RuntimeError("Lint checks found issues. Send me the output.")

# Rerun tests
test_result = subprocess.run(
    ["python", "-m", "pytest", "-v"],
    cwd=str(project_dir),
    capture_output=True,
    text=True
)

test_output = test_result.stdout + test_result.stderr
print("\n=== TEST RESULTS ===")
print(test_output)

(outputs_dir / "test_results.txt").write_text(test_output)

if test_result.returncode != 0:
    raise RuntimeError("Tests failed. Send me the output.")

print("\nSUCCESS: Lint checks and tests passed.")

=== LINT RESULTS ===
I001 [*] Import block is un-sorted or un-formatted
 --> app.py:1:1
  |
1 | / from fastapi import FastAPI, Request
2 | | from pydantic import BaseModel
3 | | from sklearn.linear_model import LogisticRegression
4 | | import time
  | |___________^
5 |
6 |   app = FastAPI(title="W12D4 End-to-End MLOps API")
  |
help: Organize imports
  |
1 + import time
2 +
3 | from fastapi import FastAPI, Request
4 | from pydantic import BaseModel
5 | from sklearn.linear_model import LogisticRegression
  - import time
6 |
  |

I001 [*] Import block is un-sorted or un-formatted
 --> tests/test_app.py:1:1
  |
1 | / from fastapi.testclient import TestClient
2 | | from app import app
  | |___________________^
3 |
4 |   client = TestClient(app)
  |
help: Organize imports
  |
1 | from fastapi.testclient import TestClient
2 +
3 | from app import app
  |

Found 2 errors.
[*] 2 fixable with `--fix`.



RuntimeError: Lint checks found issues. Send me the output.

In [14]:
# Automatically fix import formatting
import subprocess

result = subprocess.run(
    ["python", "-m", "ruff", "check", "--fix", "app.py", "tests"],
    cwd=str(project_dir),
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Ruff could not fix all issues.")

print("Import formatting fixed!")

Found 2 errors (2 fixed, 0 remaining).


Import formatting fixed!


In [16]:
# W12D4: Verify project structure before downloading

required_files = [
    "app.py",
    "Dockerfile",
    "requirements.txt",
    "requirements-dev.txt",
    "monitoring.md",
    ".gitignore",
    "tests/test_app.py",
    ".github/workflows/ci.yml",
    "outputs/test_results.txt",
    "outputs/lint_results.txt",
    "outputs/health_output.json",
    "outputs/prediction_output.json",
    "outputs/monitoring_metrics.json",
]

missing_files = []

for filename in required_files:
    file_path = project_dir / filename
    if file_path.exists():
        print("✅", filename)
    else:
        print("❌ MISSING:", filename)
        missing_files.append(filename)

if missing_files:
    print("\nPlease share the missing filenames before downloading.")
else:
    print("\nSUCCESS: All W12D4 project files are ready.")

✅ app.py
✅ Dockerfile
✅ requirements.txt
✅ requirements-dev.txt
✅ monitoring.md
✅ .gitignore
✅ tests/test_app.py
✅ .github/workflows/ci.yml
✅ outputs/test_results.txt
✅ outputs/lint_results.txt
✅ outputs/health_output.json
✅ outputs/prediction_output.json
✅ outputs/monitoring_metrics.json

SUCCESS: All W12D4 project files are ready.


In [17]:
# W12D4: Download project files individually

from google.colab import files
import time

files_to_download = [
    "app.py",
    "Dockerfile",
    "requirements.txt",
    "requirements-dev.txt",
    "README.md",
    "monitoring.md",
    ".gitignore",
    "tests/test_app.py",
    ".github/workflows/ci.yml",
    "outputs/test_results.txt",
    "outputs/lint_results.txt",
    "outputs/health_output.json",
    "outputs/prediction_output.json",
    "outputs/monitoring_metrics.json",
]

for filename in files_to_download:
    path = project_dir / filename
    if path.exists():
        print("Downloading:", filename)
        files.download(str(path))
        time.sleep(2)
    else:
        print("Skipped missing file:", filename)

print("Download requests completed.")

Downloading: app.py


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading: Dockerfile


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

Downloading: outputs/monitoring_metrics.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download requests completed.
